# Building the wiring diagram

In [1]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.append('../../')

In [2]:

import glob
import pickle
from pathlib import Path

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import matplotlib as mpl

import scipy
import pandas as pd
import seaborn as sns

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict, cross_validate, StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


from dataclasses import dataclass

import pyaldata as pyal

import tools.dsp as dsp
from tools.params import Params
import tools.dimensionality as dim
import tools.decoding as decode
import tools.subspaces as subspaces
import tools.viz.utilityTools as vizutils
import tools.dataTools as dt
from tools.params import colors
import tools.reports as reports
import tools.kinematics as kin
from tools.viz.rc_style import rc_context_talk


from tqdm import tqdm
import pickle

from joblib import Parallel, delayed

import numpy as np

In [3]:
data_dir='/data/bnd-data/raw/'

SESSIONS = [
    # "M061_2025_03_04_10_00",
    # "M061_2025_03_05_14_00",
    # "M061_2025_03_06_14_00",
    # "M063_2025_03_13_14_00",
    # "M063_2025_03_14_15_30",
    # "M062_2025_03_20_14_00",
    # "M062_2025_03_21_14_00",
    # "M078_2025_08_06_15_00",
    # "M086_2025_12_10_15_00",
    # "M103_2026_02_17_14_00",
    # "M103_2026_02_18_15_30",
    # "M103_2026_02_19_15_30",
    # "M106_2026_02_24_15_00",
    "M106_2026_02_25_15_00",
    # "M106_2026_02_26_16_00",
]


for sess in SESSIONS:
    td, _ = dsp.load_and_process_session(sess, std=0.03)
    perturb_td = pyal.restrict_to_interval(td, start_point_name='idx_sol_on', rel_start=-200, rel_end=300)
    perturb_td_shuff = perturb_td.sample(frac=1, random_state=42).reset_index(drop=False)

fields: ['values_before_camera_trigger', 'idx_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
['CP_spikes', 'MOp_spikes', 'all_spikes', 'VAL_spikes', 'SSp_spikes']


/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: values_MotSen1_X might be a time-varying field. It matches the length of CP_spikes on 99.9001996007984% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: idx_MotSen1_X might be a time-varying field. It matches the length of CP_spikes on 99.9001996007984% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: values_MotSen1_Y might be a time-varying field. It matches the length of CP_spikes on 99.9001996007984% of trials
  warnings.warn(
/home/me24/repos/pyaldata/pyaldata/utils.py:155: UserWarning: idx_MotSen1_Y might be a time-varying field. It matches the length of CP_spikes on 99.9001996007984% of trials
  warnings.warn(


Resulting CP_spikes ephys data shape is (NxT): (206, 48000)
Resulting MOp_spikes ephys data shape is (NxT): (156, 48000)
Resulting all_spikes ephys data shape is (NxT): (11, 48000)
Resulting VAL_spikes ephys data shape is (NxT): (132, 48000)
Resulting SSp_spikes ephys data shape is (NxT): (142, 48000)
add_concat_perturb_time: dropping 1 trial(s) with missing idx_sol_on
Skipped 87 trials
Otsu immobility threshold: 2.2123
Dropped 130 of 499 rows (26.05%).


In [6]:
df = pyal.load_pyaldata(data_dir + SESSIONS[0][:4] + "/" + SESSIONS[0])

fields: ['values_before_camera_trigger', 'idx_before_camera_trigger', 'values_Sol_duration', 'values_Sol_direction'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.
fields: ['values_before_camera_trigger', 'values_Sol_duration', 'idx_Sol_duration', 'values_Sol_direction', 'idx_Sol_direction', 'idx_sol_on'] could not be converted to int.


In [7]:
df

,animal,session,trial_id,trial_name,trial_length,bin_size,idx_trial_start,idx_trial_end,idx_CPI,values_before_camera_trigger,...,all_kslabel,all_spikes,VAL_chan_best,VAL_unit_guide,VAL_kslabel,VAL_spikes,SSp_chan_best,SSp_unit_guide,SSp_kslabel,SSp_spikes
0,M106,M106_2026_02_25_15_00,0,free,48000,0.01,0,47999,[],1,...,"[mua, mua, mua, mua, mua, mua, mua, mua, mua, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,...","[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 5, 5, 6, ...","[[0, 1], [0, 2], [1, 1], [1, 2], [2, 1], [2, 2...","[good, good, mua, good, mua, mua, good, good, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,...","[220, 220, 220, 221, 221, 222, 224, 224, 224, ...","[[220, 1], [220, 2], [220, 3], [221, 1], [221,...","[good, mua, good, good, good, good, mua, good,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
1,M106,M106_2026_02_25_15_00,1,intertrial,300,0.01,48000,48299,[],[],...,"[mua, mua, mua, mua, mua, mua, mua, mua, mua, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0,...","[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 5, 5, 6, ...","[[0, 1], [0, 2], [1, 1], [1, 2], [2, 1], [2, 2...","[good, good, mua, good, mua, mua, good, good, ...","[[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,...","[220, 220, 220, 221, 221, 222, 224, 224, 224, ...","[[220, 1], [220, 2], [220, 3], [221, 1], [221,...","[good, mua, good, good, good, good, mua, good,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,..."
2,M106,M106_2026_02_25_15_00,2,trial,600,0.01,48300,48899,[],[],...,"[mua, mua, mua, mua, mua, mua, mua, mua, mua, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 5, 5, 6, ...","[[0, 1], [0, 2], [1, 1], [1, 2], [2, 1], [2, 2...","[good, good, mua, good, mua, mua, good, good, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[220, 220, 220, 221, 221, 222, 224, 224, 224, ...","[[220, 1], [220, 2], [220, 3], [221, 1], [221,...","[good, mua, good, good, good, good, mua, good,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
3,M106,M106_2026_02_25_15_00,3,intertrial,100,0.01,48900,48999,[],[],...,"[mua, mua, mua, mua, mua, mua, mua, mua, mua, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 5, 5, 6, ...","[[0, 1], [0, 2], [1, 1], [1, 2], [2, 1], [2, 2...","[good, good, mua, good, mua, mua, good, good, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[220, 220, 220, 221, 221, 222, 224, 224, 224, ...","[[220, 1], [220, 2], [220, 3], [221, 1], [221,...","[good, mua, good, good, good, good, mua, good,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,..."
4,M106,M106_2026_02_25_15_00,4,trial,600,0.01,49000,49599,[],[],...,"[mua, mua, mua, mua, mua, mua, mua, mua, mua, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,...","[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 5, 5, 6, ...","[[0, 1], [0, 2], [1, 1], [1, 2], [2, 1], [2, 2...","[good, good, mua, good, mua, mua, good, good, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[220, 220, 220, 221, 221, 222, 224, 224, 224, ...","[[220, 1], [220, 2], [220, 3], [221, 1], [221,...","[good, mua, good, good, good, good, mua, good,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,M106,M106_2026_02_25_15_00,997,intertrial,500,0.01,496600,497099,[],[],...,"[mua, mua, mua, mua, mua, mua, mua, mua, mua, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,...","[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 5, 5, 5, 5, 6, ...","[[0, 1], [0, 2], [1, 1], [1, 2], [2, 1], [2, 2...","[good, good, mua, good, mua, mua, good, good, ...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0,...","[220, 220, 220, 221, 221, 222, 224, 224, 224, ...","[[220, 1], [220, 2], [220, 3], [221, 1], [221,...","[good, mua, good, good, good, good, mua, good,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
998,M106,M106_2026_02_25_15_00,998,trial,600,0.01,497100,497699,[],[],...,"[mua, mua, mua, mua, mua, mua, mua, 